# Developed Market v6 — Data Prep with Data Fixes

Re-runs the full data prep pipeline with three key fixes:
1. **Duplicate claims removed**: 117 duplicate groups (1091 extra rows) where only claim_id differs
2. **sum_insured /1000 fix**: Contents >= 1,000,000 and Buildings >= 10,000,000 divided by 1000 (user-specified thresholds, applied to both policies AND test files)
3. **Cross-market harmonization** for future Bühlmann-Strauss credibility: construction_type mapped to common standard, age_band added to match emerging market

Output: `files_v6/proc_{line}_v6.parquet` with cleaned, enriched, BF-developed ultimate_claims targets.

In [0]:
# Step 1: Import parquet files
import pandas as pd
import numpy as np
import os

BASE = '/Workspace/Users/bridget.macdonnell1@gmail.com/iaa-hackathon-2026'
DATA_DIR = f'{BASE}/data/'
V6_DIR = f'{BASE}/files_v6/'
os.makedirs(V6_DIR, exist_ok=True)

files_to_load = [
    'policies_motor', 'policies_buildings', 'policies_contents',
    'test_motor', 'test_buildings', 'test_contents',
    'claims', 'customers', 'addresses', 'weather', 'sample_submission'
]

dfs = {}
for name in files_to_load:
    path = os.path.join(DATA_DIR, f'{name}.parquet')
    dfs[name] = pd.read_parquet(path)
    print(f'{name}: {dfs[name].shape}')

print('\nAll files loaded.')

In [0]:
# Step 2: Data cleaning

# 2a. Fix exposure = (period_end - period_start + 1) / 365
for name in ['policies_motor', 'policies_buildings', 'policies_contents',
             'test_motor', 'test_buildings', 'test_contents']:
    df = dfs[name]
    ps = pd.to_datetime(df['period_start'])
    pe = pd.to_datetime(df['period_end'])
    df['exposure'] = ((pe - ps).dt.days + 1) / 365.0
    print(f'{name}: exposure fixed, mean={df["exposure"].mean():.4f}')

# 2b. Create Year field from period_start
for name in ['policies_motor', 'policies_buildings', 'policies_contents',
             'test_motor', 'test_buildings', 'test_contents']:
    dfs[name]['Year'] = pd.to_datetime(dfs[name]['period_start']).dt.year

# 2c. Construction type: normalise + harmonize to COMMON STANDARD for Bühlmann-Strauss
# Developed: BRICK/Brick/brick -> BRICK, STONE/Stone/stone -> STONE, W/w/T/t -> TIMBER, NON_STANDARD/non_standard -> NON_STANDARD
# Emerging: BRK->BRICK, STN->STONE, WOD->TIMBER, MIX->MIXED  (applied in emerging notebook)
# Common standard: BRICK, STONE, TIMBER, NON_STANDARD, MIXED
CONSTRUCTION_MAP_DEV = {
    'BRICK': 'BRICK', 'STONE': 'STONE', 'W': 'TIMBER', 'T': 'TIMBER',
    'NON_STANDARD': 'NON_STANDARD', 'TIMBER': 'TIMBER'
}
for name in ['policies_buildings', 'test_buildings']:
    df = dfs[name]
    df['construction_type'] = df['construction_type'].astype(str).str.strip().str.upper()
    df['construction_type'] = df['construction_type'].replace(CONSTRUCTION_MAP_DEV)
    print(f'{name}: construction_type = {df["construction_type"].value_counts().to_dict()}')

# 2d. Strip whitespace from all ID columns
id_cols = ['policy_id', 'customer_id', 'address_id', 'claim_id', 'row_id']
for name, df in dfs.items():
    for col in id_cols:
        if col in df.columns:
            df[col] = df[col].astype(str).str.strip()

# 2e. Check for full row duplicates in policy files
for name in ['policies_motor', 'policies_buildings', 'policies_contents']:
    df = dfs[name]
    n_before = len(df)
    df.drop_duplicates(inplace=True)
    print(f'{name}: {n_before} -> {len(df)} rows (dropped {n_before - len(df)} full duplicates)')

# 2f. Zero-pad motor customer_id to 8 chars
for name in ['policies_motor', 'test_motor']:
    df = dfs[name]
    lens = df['customer_id'].str.len().value_counts()
    if lens.nunique() > 1:
        df['customer_id'] = df['customer_id'].str.zfill(8)
        print(f'{name}: zero-padded customer_id to 8 chars')
cust_lens = dfs['customers']['customer_id'].str.len().value_counts()
if cust_lens.nunique() > 1:
    dfs['customers']['customer_id'] = dfs['customers']['customer_id'].str.zfill(8)
    print('customers: zero-padded customer_id')

# 2g. Drop compulsory_excess if constant per line
for name in ['policies_motor', 'policies_buildings', 'policies_contents']:
    df = dfs[name]
    if 'compulsory_excess' in df.columns and df['compulsory_excess'].nunique() == 1:
        df.drop('compulsory_excess', axis=1, inplace=True)
        print(f'{name}: dropped constant compulsory_excess')

# 2h. SUM_INSURED FIX (user-specified thresholds, applied to policies AND test)
# Contents: sum_insured >= 1,000,000 -> divide by 1000
# Buildings: sum_insured >= 10,000,000 -> divide by 1000
# Flag any that might be genuine large sums (not x1000 errors)
print('\n=== SUM_INSURED FIX (v6 thresholds) ===')
for name in ['policies_buildings', 'policies_contents', 'test_buildings', 'test_contents']:
    df = dfs[name]
    if 'sum_insured' not in df.columns:
        continue
    line = 'buildings' if 'buildings' in name else 'contents'
    threshold = 10_000_000 if line == 'buildings' else 1_000_000
    flagged = df['sum_insured'] >= threshold
    n_flagged = flagged.sum()
    if n_flagged > 0:
        # Flag potentially genuine large values (check if /1000 would still be above 95th percentile)
        p95 = df.loc[~flagged, 'sum_insured'].quantile(0.95)
        corrected = df.loc[flagged, 'sum_insured'] / 1000
        # If corrected value is still > 2x p95, it might be genuine
        potentially_genuine = (corrected > 2 * p95).sum()
        df.loc[flagged, 'sum_insured'] = df.loc[flagged, 'sum_insured'] / 1000
        print(f'{name}: {n_flagged} values >= {threshold:,} divided by 1000'
              f' ({potentially_genuine} might be genuine large sums - corrected value still > 2x p95={p95:.0f})')
    else:
        print(f'{name}: no values >= {threshold:,}')

# 2i. DEDUPLICATE CLAIMS (rows where only claim_id differs)
print('\n=== DEDUPLICATE CLAIMS ===')
claims = dfs['claims']
non_id_cols = [c for c in claims.columns if c != 'claim_id']
dup_mask = claims.duplicated(subset=non_id_cols, keep=False)
n_dups = dup_mask.sum()
if n_dups > 0:
    grp_sizes = claims[dup_mask].groupby(non_id_cols).size()
    n_groups = len(grp_sizes)
    extra_rows = n_dups - n_groups
    # Keep first occurrence
    before = len(claims)
    claims = claims.drop_duplicates(subset=non_id_cols, keep='first').reset_index(drop=True)
    after = len(claims)
    print(f'Duplicate groups: {n_groups}, extra rows removed: {extra_rows} ({before} -> {after} claims)')
    dfs['claims'] = claims
else:
    print('No duplicate claims found')

# 2j. ADD AGE_BAND to developed market (matching emerging: 17-24, 25-34, 35-49, 50-64, 65+)
def to_age_band(age):
    if pd.isna(age): return None
    if age <= 24: return '17-24'
    elif age <= 34: return '25-34'
    elif age <= 49: return '35-49'
    elif age <= 64: return '50-64'
    else: return '65+'

for name in ['policies_motor', 'test_motor']:
    if 'driver_age' in dfs[name].columns:
        dfs[name]['driver_age_band'] = dfs[name]['driver_age'].apply(to_age_band)
        print(f'{name}: added driver_age_band')
for name in ['policies_buildings', 'policies_contents', 'test_buildings', 'test_contents']:
    if 'policyholder_age' in dfs[name].columns:
        dfs[name]['policyholder_age_band'] = dfs[name]['policyholder_age'].apply(to_age_band)
        print(f'{name}: added policyholder_age_band')

print('\nStep 2 complete.')

In [0]:
# Step 3: Join enrichment data to policy files
customers_clean = dfs['customers'].drop(columns=['surname', 'date_of_birth'], errors='ignore')
addresses_clean = dfs['addresses'].copy()

weather = dfs['weather'].copy()
weather_agg = weather.groupby(['address_id', 'year'], as_index=False).mean(numeric_only=True)
weather_agg.rename(columns={'year': 'Year'}, inplace=True)
print(f'Weather: {len(weather)} -> {len(weather_agg)} rows')

policy_files = {
    'motor': 'policies_motor',
    'buildings': 'policies_buildings', 
    'contents': 'policies_contents'
}

for line, fname in policy_files.items():
    df = dfs[fname]
    n_before = len(df)
    cols_before = df.shape[1]
    df = df.merge(customers_clean, on='customer_id', how='left', suffixes=('', '_cust'))
    df = df.merge(addresses_clean, on='address_id', how='left', suffixes=('', '_addr'))
    df = df.merge(weather_agg, on=['address_id', 'Year'], how='left', suffixes=('', '_weather'))
    dfs[fname] = df
    print(f'{line}: {n_before} -> {len(df)} rows, {cols_before} -> {df.shape[1]} cols')
    for ref_name, ref_df in [('customers', customers_clean), ('addresses', addresses_clean), ('weather', weather_agg)]:
        join_col = 'customer_id' if ref_name == 'customers' else 'address_id'
        matched = df[join_col].isin(ref_df[join_col]).sum()
        print(f'  {ref_name}: {matched}/{len(df)} matched ({100*matched/len(df):.1f}%)')

In [0]:
# Step 4: Claims processing
claims = dfs['claims']
claims['incurred_net_of_recoveries'] = claims['incurred_to_date'] - claims['recoveries']
print('=== Claims status ===')
print(claims['status'].value_counts().to_dict())
declined_mask = claims['status'].str.lower().isin(['declined', 'repudiated'])
print(f'Declined/repudiated: {declined_mask.sum()} (excluded)')
claims_active = claims[~declined_mask].copy()
print(f'Active claims: {len(claims_active)}')
dfs['claims_active'] = claims_active

In [0]:
# Step 5: BF development to ultimate
claims_active = dfs['claims_active']
claims_active['loss_year'] = pd.to_datetime(claims_active['loss_date']).dt.year
claims_active['report_date_dt'] = pd.to_datetime(claims_active['report_date'], errors='coerce')
claims_active['report_year'] = claims_active['report_date_dt'].dt.year
claims_active['dev_lag'] = claims_active['report_year'] - claims_active['loss_year']

triangle_raw = claims_active.groupby(['loss_year', 'dev_lag']).size().unstack(fill_value=0)
count_triangle = triangle_raw.cumsum(axis=1)
print('=== Claim Count Triangle ===')
print(count_triangle)

DEV_FACTORS = {
    2021: {'bi': 1.00, 'tpp': 1.00, 'short_motor': 1.00, 'subsidence': 1.00, 'buildings_other': 1.00, 'contents': 1.00},
    2022: {'bi': 1.01, 'tpp': 1.01, 'short_motor': 1.01, 'subsidence': 1.05, 'buildings_other': 1.02, 'contents': 1.01},
    2023: {'bi': 1.06, 'tpp': 1.01, 'short_motor': 1.01, 'subsidence': 1.15, 'buildings_other': 1.02, 'contents': 1.01},
    2024: {'bi': 1.82, 'tpp': 1.08, 'short_motor': 1.08, 'subsidence': 1.47, 'buildings_other': 1.08, 'contents': 1.04},
}

def get_dev_factor(peril, line, loss_year):
    p = str(peril).lower()
    if line == 'motor':
        if 'bodily' in p or 'injury' in p: tail = 'bi'
        elif 'third_party' in p: tail = 'tpp'
        else: tail = 'short_motor'
    elif line == 'buildings':
        tail = 'subsidence' if 'subsidence' in p else 'buildings_other'
    else: tail = 'contents'
    return DEV_FACTORS.get(loss_year, {}).get(tail, 1.0)

claims_active['dev_factor'] = claims_active.apply(lambda r: get_dev_factor(r['peril'], r['line'], r['loss_year']), axis=1)
claims_active['ultimate_claims'] = claims_active['incurred_net_of_recoveries'] * claims_active['dev_factor']
dfs['claims_active'] = claims_active
print(f'\nTotal ultimate claims: {claims_active["ultimate_claims"].sum():,.2f}')

In [0]:
# Step 5b: Match claims to policy rows and aggregate ultimate_claims by row_id
claims_active = dfs['claims_active']
claims_active['loss_date_dt'] = pd.to_datetime(claims_active['loss_date'])

for line, fname in policy_files.items():
    df = dfs[fname].copy()
    df['_ps_dt'] = pd.to_datetime(df['period_start'])
    df['_pe_dt'] = pd.to_datetime(df['period_end'])
    line_claims = claims_active[claims_active['line'] == line].copy()
    print(f'\n=== {line}: {len(line_claims)} claims -> {len(df)} policy rows ===')
    merged = line_claims.merge(df[['row_id', 'policy_id', '_ps_dt', '_pe_dt', 'Year']], on='policy_id', how='inner')
    merged = merged[merged['loss_date_dt'].dt.year == merged['Year']]
    mask = (merged['loss_date_dt'] >= merged['_ps_dt']) & (merged['loss_date_dt'] <= merged['_pe_dt'])
    matched = merged[mask].copy()
    matched = matched.sort_values('row_id').drop_duplicates('claim_id', keep='first')
    print(f'  Matched: {len(matched)} claims ({len(line_claims) - len(matched)} unmatched)')
    agg = matched.groupby('row_id').agg(
        ultimate_claims=('ultimate_claims', 'sum'),
        claim_count=('claim_id', 'count'),
        reported_incurred=('incurred_net_of_recoveries', 'sum'),
    ).reset_index()
    df = df.merge(agg, on='row_id', how='left')
    df['ultimate_claims'] = df['ultimate_claims'].fillna(0)
    df['claim_count'] = df['claim_count'].fillna(0).astype(int)
    df['reported_incurred'] = df['reported_incurred'].fillna(0)
    df.drop(['_ps_dt', '_pe_dt'], axis=1, inplace=True)
    dfs[fname] = df
    n_with = (df['claim_count'] > 0).sum()
    print(f'  Rows with claims: {n_with} ({100*n_with/len(df):.1f}%)')
    print(f'  Total ultimate: {df["ultimate_claims"].sum():,.2f}, mean={df["ultimate_claims"].mean():.4f}')

In [0]:
# Step 6: Save v6 proc files + harmonization summary
for line, fname in policy_files.items():
    df = dfs[fname]
    out_path = os.path.join(V6_DIR, f'proc_{line}_v6.parquet')
    df.to_parquet(out_path, index=False)
    print(f'Saved: {out_path} ({len(df)} rows, {df.shape[1]} cols)')
    print(f'  Target mean={df["ultimate_claims"].mean():.4f}, sum={df["ultimate_claims"].sum():,.2f}')
    print(f'  Rows with claims: {(df["claim_count"]>0).sum()} ({100*(df["claim_count"]>0).mean():.1f}%)')
    # Verify sum_insured fix
    if 'sum_insured' in df.columns:
        if line == 'buildings':
            n_bad = (df['sum_insured'] >= 10_000_000).sum()
        else:
            n_bad = (df['sum_insured'] >= 1_000_000).sum()
        print(f'  sum_insured check: {n_bad} values still >= threshold (should be 0)')
    # Verify harmonization columns
    if line == 'buildings':
        print(f'  construction_type: {sorted(df["construction_type"].dropna().unique().tolist())}')
    age_band_col = 'driver_age_band' if line == 'motor' else 'policyholder_age_band'
    if age_band_col in df.columns:
        print(f'  {age_band_col}: {sorted(df[age_band_col].dropna().unique().tolist())}')

# Save harmonization summary
harmonization = {
    'construction_type_dev': 'BRICK, STONE, TIMBER, NON_STANDARD (mapped from raw: W/T->TIMBER)',
    'construction_type_emg': 'BRICK, STONE, TIMBER, MIXED (mapped from raw: BRK/STN/WOD/MIX)',
    'common_standard': 'BRICK, STONE, TIMBER, NON_STANDARD, MIXED',
    'age_band': '17-24, 25-34, 35-49, 50-64, 65+ (added to developed, already in emerging)',
    'dev_only_features': 'credit_band, date_of_birth, telematics_score, annual_mileage',
    'emg_only_features': 'gender',
    'note': 'For Bühlmann-Strauss credibility: use common columns (construction_type, age_band, sum_insured, exposure, Year, line) as matching keys. credit_band and gender are market-specific and cannot be directly mapped.',
}
import json
with open(os.path.join(V6_DIR, 'harmonization_summary.json'), 'w') as f:
    json.dump(harmonization, f, indent=2)
print(f'\nSaved harmonization summary to {V6_DIR}harmonization_summary.json')
print('\n=== v6 Data Prep Complete ===')